# Retro-Dip Point Selection

Scan 1-minute candle VWAP for **retro-dip** mean-reversion structures
(past anchor premium → interim dip → partial recovery), cluster contiguous
signals, and explore the resulting clusters on an interactive price chart.

In [ ]:
%pip install numpy numba pandas pyarrow requests plotly ipywidgets anywidget

In [ ]:
import os
import sys

# Clone this project's repo so the local `packages` are importable in a remote
# Jupyter server. This notebook lives on a feature branch, so clone that branch.
REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"
BRANCH_NAME = "claude/keen-thompson-ut54oo"

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH_NAME} {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if os.path.isdir(REPO_PATH) and REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

# Fallback for running from inside the repo: walk up to the folder holding
# `packages/` and add it to sys.path so `from packages.retro_dip import ...` works.
_p = os.path.abspath(os.getcwd())
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "packages")) and _p not in sys.path:
        sys.path.append(_p)
        break
    _p = os.path.dirname(_p)

## Parameters

In [ ]:
# --- Asset & retro-dip parameters ---
asset_name           = "btcusdt"   # lowercase asset folder name
max_lookback_window  = 1440        # max backward search depth (minutes / candles)
target_past_premium  = 0.0100      # required past-anchor premium (100 bps)
min_interim_drawdown = 0.0020      # minimum interim dip below current (20 bps)
max_interim_drawdown = 0.0050      # maximum interim dip below current (50 bps)

# --- Study date range (UTC), inclusive ---
start = "2026-03-01 00:00"
end   = "2026-03-07 23:59"

## Load candle data

Candle data is cached under `./data/` as monthly parquet files. If a required
month is missing it is downloaded from the HuggingFace candles dataset and saved
for subsequent runs. The load window is extended `max_lookback_window` minutes
before `start` so that every in-range candle has a full look-back behind it.

In [ ]:
import io
import requests
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from datetime import datetime, timedelta, timezone

DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)
CANDLES_REPO = "https://huggingface.co/datasets/payamdavaee/candles/resolve/main"


def _month_iter(d0, d1):
    """Yield (year, month) tuples from d0's month through d1's month inclusive."""
    y, m = d0.year, d0.month
    while (y < d1.year) or (y == d1.year and m <= d1.month):
        yield y, m
        m += 1
        if m > 12:
            m = 1
            y += 1


def load_candles(asset, load_start, load_end, columns=("ts", "vwap")):
    """Load 1-minute candles for [load_start, load_end], caching months to ./data/."""
    frames = []
    for y, m in _month_iter(load_start, load_end):
        fname = f"{asset}-1m-{y:04d}-{m:02d}.parquet"
        local = os.path.join(DATA_DIR, fname)
        if not os.path.exists(local):
            url = f"{CANDLES_REPO}/{asset}/{fname}"
            print("downloading", url)
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
            with open(local, "wb") as fh:
                fh.write(resp.content)
        else:
            print("cached", local)
        frames.append(pq.read_table(local, columns=list(columns)).to_pandas())

    df = pd.concat(frames).sort_values("ts").reset_index(drop=True)
    ts = df["ts"].astype("int64")
    s = int(load_start.timestamp() * 1000)
    e = int(load_end.timestamp() * 1000)
    df = df[(ts >= s) & (ts <= e)].reset_index(drop=True)
    return df


dt_start = datetime.fromisoformat(start).replace(tzinfo=timezone.utc)
dt_end   = datetime.fromisoformat(end).replace(tzinfo=timezone.utc)
load_start = dt_start - timedelta(minutes=max_lookback_window)
load_end   = dt_end

df = load_candles(asset_name, load_start, load_end)

prices = df["vwap"].to_numpy(dtype=np.float64)
ts     = df["ts"].astype("int64").to_numpy()
times  = pd.to_datetime(ts, unit="ms", utc=True)

print(f"Loaded {len(df):,} candles from {times[0]} to {times[-1]}")

## Run signal detection

Run the retro-dip scanner over the loaded VWAP array, then report how many
candle points fall inside the study date range and how many of them produced a
valid signal.

In [ ]:
from packages.retro_dip import scan_retro_dip_signals

signals = scan_retro_dip_signals(
    prices,
    max_lookback_window,
    target_past_premium,
    min_interim_drawdown,
    max_interim_drawdown,
)

start_ms = int(dt_start.timestamp() * 1000)
end_ms   = int(dt_end.timestamp() * 1000)

# Candle points inside the study date range.
in_range = (ts >= start_ms) & (ts <= end_ms)
n_candles_in_range = int(in_range.sum())

# Keep only signals whose current candle lies inside the study date range.
sig_idx = signals[:, 0].astype(np.int64)
sig_in_range = (ts[sig_idx] >= start_ms) & (ts[sig_idx] <= end_ms)
signals = np.ascontiguousarray(signals[sig_in_range])

print(f"Candle points in date range : {n_candles_in_range:,}")
print(f"Signal points in result     : {signals.shape[0]:,}")
if n_candles_in_range:
    print(f"Hit rate                    : {100 * signals.shape[0] / n_candles_in_range:.3f}%")

## Cluster contiguous signals

Merge strictly contiguous signals into clusters (each represented by its
earliest point) and summarise the cluster count and the distribution of cluster
lengths.

In [ ]:
from packages.retro_dip import cluster_retro_dip_signals

clusters = cluster_retro_dip_signals(signals)
lengths = clusters[:, 3]

print(f"Number of clusters        : {clusters.shape[0]:,}")
print(f"Total clustered signals   : {int(lengths.sum()):,}")
print()

if clusters.shape[0]:
    stats = pd.Series(lengths, name="cluster_length").describe()
    print("Cluster length statistics:")
    print(stats.to_string())
    print()
    print(f"median length : {np.median(lengths):.2f}")
    print(f"max length    : {int(lengths.max())}")
    print(f"min length    : {int(lengths.min())}")

## Interactive cluster explorer

Price (VWAP) over time. Each cluster's contiguous span is drawn as a red
horizontal line at the cluster's starting price. Use **Previous** / **Next** to
jump the chart view to the start of the previous / next cluster.

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

c_start = clusters[:, 0].astype(np.int64)
c_len   = clusters[:, 3].astype(np.int64)
c_end   = c_start + c_len - 1

# Build all cluster spans as a single trace using None separators (efficient).
seg_x, seg_y = [], []
for s_i, e_i in zip(c_start, c_end):
    seg_x += [times[s_i], times[e_i], None]
    seg_y += [prices[s_i], prices[s_i], None]

fig = go.FigureWidget()
fig.add_trace(go.Scattergl(x=times, y=prices, mode="lines",
                           name="VWAP", line=dict(color="#1f77b4", width=1)))
fig.add_trace(go.Scattergl(x=seg_x, y=seg_y, mode="lines",
                           name="Cluster span", line=dict(color="red", width=6)))
fig.update_layout(title="Retro-dip clusters", xaxis_title="Time",
                  yaxis_title="VWAP", height=520, hovermode="x unified",
                  legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))

view_minutes = max_lookback_window  # half-width of the view window
state = {"i": 0}


def goto(i):
    if clusters.shape[0] == 0:
        return
    i = max(0, min(clusters.shape[0] - 1, i))
    state["i"] = i
    cs = c_start[i]
    x_left  = times[max(0, cs - view_minutes)]
    x_right = times[min(len(times) - 1, cs + view_minutes)]
    with fig.batch_update():
        fig.layout.xaxis.range = [x_left, x_right]
        fig.layout.title.text = (
            f"Cluster {i + 1}/{clusters.shape[0]} \u2014 "
            f"start {times[cs]}, length {int(c_len[i])}"
        )


btn_prev = widgets.Button(description="\u25c0 Previous")
btn_next = widgets.Button(description="Next \u25b6")
label = widgets.Label()
btn_prev.on_click(lambda b: goto(state["i"] - 1))
btn_next.on_click(lambda b: goto(state["i"] + 1))

if clusters.shape[0]:
    goto(0)
else:
    print("No clusters to display.")

display(widgets.HBox([btn_prev, btn_next]))
display(fig)